###  Goal of this step
Transform raw sales data into a machine-learning ready time-series dataset for forecasting.

In [1]:
import pandas as pd
import numpy as np


Load Dataset

In [2]:
df = pd.read_csv("data/raw/Super store Sales.csv")

In [3]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df = df.dropna(subset=["Order Date", "Sales","Postal Code"])
df = df.sort_values("Order Date")

Monthly Aggregation (Forecast Base)

We forecast monthly sales, not raw transactions.

In [4]:
monthly_df = (
    df
    .set_index("Order Date")
    .resample("M")["Sales"]
    .sum()
    .reset_index()
)

monthly_df.head()


C:\Users\K ADITHYA\AppData\Local\Temp\ipykernel_2264\1249548904.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample("M")["Sales"]


,Order Date,Sales
0,2015-01-31,14205.707
1,2015-02-28,4519.892
2,2015-03-31,55205.797
3,2015-04-30,27906.855
4,2015-05-31,23644.303


Extract Time Features

These help ML models understand seasonality.

In [5]:
monthly_df["month"] = monthly_df["Order Date"].dt.month
monthly_df["year"] = monthly_df["Order Date"].dt.year
#“Month captures seasonality, year captures long-term trend.”

Create Lag Features

Lag features let the model learn from past sales.

1-Month Lag

In [6]:
monthly_df["lag_1"] = monthly_df["Sales"].shift(1)


3-Month Lag

In [7]:
monthly_df["lag_3"] = monthly_df["Sales"].shift(3)


“Lag features convert a time-series problem into a supervised learning problem.”

Rolling Averages (Trend Smoothing)

In [8]:
monthly_df["rolling_3_mean"] = (
    monthly_df["Sales"]
    .rolling(window=3)
    .mean()
)


📌 Why:

Reduces noise

Captures local trend

Handle NaNs Created by Lags

Lag & rolling features always create NaNs.

In [9]:
monthly_df = monthly_df.dropna()


Final Feature Set Preview

In [10]:
monthly_df.head()


,Order Date,Sales,month,year,lag_1,lag_3,rolling_3_mean
3,2015-04-30,27906.8550,4,2015,55205.7970,14205.707,29210.848000
4,2015-05-31,23644.3030,5,2015,27906.8550,4519.892,35585.651667
5,2015-06-30,34322.9356,6,2015,23644.3030,55205.797,28624.697867
6,2015-07-31,33781.5430,7,2015,34322.9356,27906.855,30582.927200
7,2015-08-31,27117.5365,8,2015,33781.5430,23644.303,31740.671700


Save Processed Dataset

In [11]:
monthly_df.to_csv("data/processed/forecast_data.csv", index=False)


Feature Engineering Summary

Aggregated transactional data to monthly sales

Extracted temporal features for seasonality

Created lag features to capture historical dependency

Added rolling averages for trend smoothing

Prepared clean supervised learning dataset